# CIFAR-10 CNN training for hls4ml (v0.2)

Adapted from ICTP MNIST CNN training notebook, retargeted to CIFAR-10.

## Pipeline (same structure as upstream ICTP)

1. **Teacher**: full-precision Keras CNN trained on CIFAR-10 to get a strong baseline.
2. **Student**: small QKeras quantised CNN trained via Knowledge Distillation from the teacher.
3. **Save** the distilled student model. That is what we will convert with hls4ml and synthesise into the FPGA in the next stage.

## Differences from MNIST upstream

| | MNIST (upstream) | CIFAR-10 (this notebook) |
|---|---|---|
| Input | 28x28x1 grayscale | 32x32x3 RGB |
| Classes | 10 digits | 10 object categories |
| Teacher first Conv | 16 filters | 32 filters (RGB needs more channels) |
| Student first Conv | 2 filters | 8 filters (CIFAR is harder) |
| Dataset source | CSV file | `tf.keras.datasets.cifar10` (auto-downloaded) |

In [ ]:
# === Libraries ===
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.datasets import cifar10

import qkeras
from qkeras import QConv2D, QDense, QActivation, quantized_bits, quantized_relu

from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print('QKeras:', qkeras.__version__)
print('Devices:', tf.config.list_physical_devices())

In [ ]:
# === Reproducibility ===
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

## 1. Dataset loading and preprocessing

CIFAR-10 ships with Keras and downloads automatically (~170 MB the first time, cached afterwards in `~/.keras/datasets/`).

In [ ]:
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = cifar10.load_data()

print('Raw shapes:')
print('  x_train:', x_train_raw.shape, 'dtype:', x_train_raw.dtype, 'min/max:', x_train_raw.min(), x_train_raw.max())
print('  y_train:', y_train_raw.shape, 'dtype:', y_train_raw.dtype)
print('  x_test :', x_test_raw.shape)
print('  y_test :', y_test_raw.shape)

In [ ]:
# Normalise to [0, 1] float32 — same pattern as the MNIST upstream
x_train = x_train_raw.astype('float32') / 255.0
x_test  = x_test_raw.astype('float32')  / 255.0

# Labels: one-hot for training, integer for evaluation
NUM_CLASSES = 10
y_train = keras.utils.to_categorical(y_train_raw, NUM_CLASSES)
y_test  = keras.utils.to_categorical(y_test_raw,  NUM_CLASSES)

y_test_int = y_test_raw.flatten()

print('After preprocessing:')
print('  x_train:', x_train.shape, x_train.dtype, x_train.min(), x_train.max())
print('  y_train:', y_train.shape, y_train.dtype)

# CIFAR-10 class names for later plots
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
# Visualise a few samples
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i])
    ax.set_title(CLASS_NAMES[int(np.argmax(y_train[i]))])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Teacher model (full-precision baseline)

Topology mirrors the upstream MNIST teacher but with more channels in each layer to handle RGB and the higher visual complexity of CIFAR-10.

| Layer | MNIST upstream | This CIFAR-10 teacher |
|---|---|---|
| Conv1 | 16 @ 3x3 | 32 @ 3x3 |
| Conv2 | 32 @ 3x3 | 64 @ 3x3 |
| Conv3 | 32 @ 3x3 | 64 @ 3x3 |
| Dense | 64 | 128 |

This teacher is **not** what runs on the FPGA. It is the soft-target source for the student.

In [ ]:
def build_teacher():
    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),

        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),   # 32x32 -> 16x16

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),   # 16x16 -> 8x8

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),   # 8x8 -> 4x4

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ])
    return model

teacher = build_teacher()
teacher.summary()

In [ ]:
# Compile and train the teacher
teacher.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

EPOCHS_TEACHER = 20
BATCH = 128

early_stop = callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True)

t0 = time.time()
hist_t = teacher.fit(
    x_train, y_train,
    batch_size=BATCH,
    epochs=EPOCHS_TEACHER,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=2,
)
print(f'Teacher training time: {time.time()-t0:.1f}s')

In [ ]:
# Evaluate teacher on test set
loss_t, acc_t = teacher.evaluate(x_test, y_test, verbose=0)
print(f'Teacher test accuracy (float32): {acc_t*100:.2f} %')

os.makedirs('models', exist_ok=True)
teacher.save('models/teacher_cifar10.h5')
print('Teacher saved to models/teacher_cifar10.h5')

## 3. Student model (quantised, for FPGA deployment)

Same structural pattern as the MNIST distilled student but slightly larger so CIFAR-10 doesn't collapse to chance accuracy.

| Layer | MNIST student | This CIFAR-10 student |
|---|---|---|
| QConv1 | 2 filters | 8 filters |
| QConv2 | 4 filters | 16 filters |
| QDense | 8 units | 32 units |

All `QConv2D` / `QDense` weights and activations are **8-bit fixed-point** (`quantized_bits(8,3)`). This roughly mimics what an INT8 DPU would do.

Parameter count target: ~5-15 K — small enough to fit fully on-chip on ZU3EG.

In [ ]:
def build_student():
    # 8 bits total, 3 integer bits, signed -> range ~ [-4, 4)
    qbits = quantized_bits(bits=8, integer=3, alpha=1)
    qrelu = quantized_relu(bits=8, integer=3)

    model = models.Sequential([
        layers.Input(shape=(32, 32, 3), name='conv1_input'),

        QConv2D(8,  (3, 3), padding='same',
                kernel_quantizer=qbits, bias_quantizer=qbits, name='conv1'),
        QActivation(qrelu, name='act1'),
        layers.MaxPooling2D((2, 2), name='pool1'),  # 32 -> 16

        QConv2D(16, (3, 3), padding='same',
                kernel_quantizer=qbits, bias_quantizer=qbits, name='conv2'),
        QActivation(qrelu, name='act2'),
        layers.MaxPooling2D((2, 2), name='pool2'),  # 16 -> 8

        layers.Flatten(name='flatten'),
        QDense(32,
               kernel_quantizer=qbits, bias_quantizer=qbits, name='fc1'),
        QActivation(qrelu, name='act_fc1'),
        QDense(NUM_CLASSES,
               kernel_quantizer=qbits, bias_quantizer=qbits, name='output'),
        layers.Activation('softmax', name='softmax'),
    ])
    return model

student = build_student()
student.summary()

### Knowledge Distillation

Train the student with a combined loss:
- hard loss against ground-truth labels (categorical cross-entropy)
- soft loss against the teacher's softmax (KL divergence with temperature)

Same idea as `distillationClassKeras.py` in the ICTP `common/` folder.

In [ ]:
class Distiller(tf.keras.Model):
    def __init__(self, student, teacher):
        super().__init__()
        self.student = student
        self.teacher = teacher

    def compile(self, optimizer, student_loss_fn, distill_loss_fn,
                alpha=0.1, temperature=4):
        super().compile(optimizer=optimizer, metrics=['accuracy'])
        self.student_loss_fn = student_loss_fn
        self.distill_loss_fn = distill_loss_fn
        self.alpha = alpha
        self.temperature = temperature

    def train_step(self, data):
        x, y = data
        teacher_pred = self.teacher(x, training=False)
        with tf.GradientTape() as tape:
            student_pred = self.student(x, training=True)
            student_loss = self.student_loss_fn(y, student_pred)
            t = self.temperature
            distill_loss = self.distill_loss_fn(
                tf.nn.softmax(tf.math.log(teacher_pred + 1e-9) / t, axis=1),
                tf.nn.softmax(tf.math.log(student_pred + 1e-9) / t, axis=1),
            )
            loss = self.alpha * student_loss + (1 - self.alpha) * distill_loss
        grads = tape.gradient(loss, self.student.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.student.trainable_variables))
        self.compiled_metrics.update_state(y, student_pred)
        return {**{m.name: m.result() for m in self.metrics},
                'student_loss': student_loss, 'distill_loss': distill_loss}

    def test_step(self, data):
        x, y = data
        student_pred = self.student(x, training=False)
        student_loss = self.student_loss_fn(y, student_pred)
        self.compiled_metrics.update_state(y, student_pred)
        return {**{m.name: m.result() for m in self.metrics},
                'student_loss': student_loss}

In [ ]:
distiller = Distiller(student=student, teacher=teacher)
distiller.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    student_loss_fn=tf.keras.losses.CategoricalCrossentropy(),
    distill_loss_fn=tf.keras.losses.KLDivergence(),
    alpha=0.3,
    temperature=4,
)

EPOCHS_STUDENT = 30
t0 = time.time()
hist_s = distiller.fit(
    x_train, y_train,
    batch_size=BATCH,
    epochs=EPOCHS_STUDENT,
    validation_split=0.1,
    verbose=2,
)
print(f'Student training time: {time.time()-t0:.1f}s')

In [ ]:
# Evaluate student
y_pred_probs = student.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
acc_s = accuracy_score(y_test_int, y_pred)
print(f'Student test accuracy (QKeras, software): {acc_s*100:.2f} %')
print(f'Teacher test accuracy (float32):           {acc_t*100:.2f} %')
print(f'Distillation gap: {(acc_t-acc_s)*100:.2f} percentage points')

student.save('models/distilled_cifar10.h5')
print('Student saved to models/distilled_cifar10.h5')

In [ ]:
# Confusion matrix on test set
cm = confusion_matrix(y_test_int, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cbar=False, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Student CIFAR-10 confusion matrix (software, {acc_s*100:.2f} % acc)')
plt.tight_layout()
plt.show()

## Next step

The student model is saved at `models/distilled_cifar10.h5`. Move on to the
`02.hls4ml/cifar10-hls4ml.ipynb` notebook to convert it into an HLS IP core.